In [15]:
from typing import Union

import pandas as pd
import numpy as np
import sklearn as sk
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


ds = load_dataset("sh0416/ag_news")
train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])


In [14]:
print("Example Training Entry:")
print(train_df.head(1))
print("\nDistribution of Training Labels:")
print(train_df.groupby("label").size())

print("\nExample Testing Entry:")
print(test_df.head(1))
print("\nDistribution of Testing Labels:")
print(test_df.groupby("label").size())

Example Training Entry:
   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   

                                         description  
0  Reuters - Short-sellers, Wall Street's dwindli...  

Distribution of Training Labels:
label
1    30000
2    30000
3    30000
4    30000
dtype: int64

Example Testing Entry:
   label                              title  \
0      3  Fears for T N pension after talks   

                                         description  
0  Unions representing workers at Turner   Newall...  

Distribution of Testing Labels:
label
1    1900
2    1900
3    1900
4    1900
dtype: int64


In [20]:
total_train_np = train_df.to_numpy()
train, val = train_test_split(
    total_train_np,
    test_size=0.1,
    random_state=SEED,
)
test = test_df.to_numpy()

train_examples = train[:, 1]
train_labels = train[:, 0]
val_examples = val[:, 1]
val_labels = val[:, 0]
test_examples = test[:, 1]
test_labels = test[:, 0]

print("Num Training Examples:", len(train))
print("Num Validation Examples:", len(val))
print("Num Testing Examples:", len(test))


Num Training Examples: 108000
Num Validation Examples: 12000
Num Testing Examples: 7600


In [22]:
from torch.utils.data import dataset
from typing import Union

# should already be tokenized
class NewsDataset(dataset.Dataset):
    def __init__(self, examples: Union[torch.Tensor, np.ndarray], labels: Union[torch.Tensor, np.ndarray]):
        self.examples = examples if isinstance(examples, torch.Tensor) else torch.Tensor(examples)
        self.labels = labels if isinstance(labels, torch.Tensor) else torch.Tensor(labels)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx], self.labels[idx]

In [ ]:
# should take in the train set and then create a vocab.
# has a process dataset and match methods to retrieve vocab and map it to an integer
# needs to have EOS, SOS, UNK, and PAD
# should first go through all descriptions and titles, count all words. if freq>2 in the dataset, add to vocab
class Tokenizer:
    pass